In [45]:
import openai
import langchain
import pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from pinecone import Pinecone
from pinecone import ServerlessSpec


In [46]:
from dotenv import load_dotenv
load_dotenv()

True

In [47]:
import os

In [48]:
##Lets read the documents
def read_docs(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

In [49]:
doc=read_docs('documents/')
len(doc)

78

In [50]:
## Devide the docs into chunks
def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc = text_splitter.split_documents(docs)
    return docs

In [51]:
documents = chunk_data(docs=doc)
len(documents)

78

In [52]:
##Check if OPENAI_API_KEY is set
print("OPENAI_API_KEY is set:", "OPENAI_API_KEY" in os.environ)
if "OPENAI_API_KEY" in os.environ:
    print("API Key found (first 10 chars):", os.environ['OPENAI_API_KEY'][:10] + "...")
else:
    print("OPENAI_API_KEY is NOT set")

OPENAI_API_KEY is set: True
API Key found (first 10 chars): sk-proj-od...


In [54]:
##Embedding Technique of OPENAI
embeddings = OpenAIEmbeddings(api_key=os.environ['OPENAI_API_KEY'])
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x13cae4ac0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x138ae3640>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [55]:
vectors=embeddings.embed_query("Hi")
vectors

[-0.0363546647131443,
 -0.007160174660384655,
 -0.03377465531229973,
 -0.02864069864153862,
 -0.02679038979113102,
 0.03458253666758537,
 -0.0124635249376297,
 -0.007837752811610699,
 0.0019187183352187276,
 -0.002667963271960616,
 0.02471856400370598,
 -0.0024643640499562025,
 -0.005788730923086405,
 -0.0029920930974185467,
 0.00666176388040185,
 -0.0030214113648980856,
 0.03385283797979355,
 -0.0015408382751047611,
 0.021057037636637688,
 -0.00903654471039772,
 -0.02173461578786373,
 0.010365639813244343,
 0.006283883936703205,
 0.007081992458552122,
 -0.012261554598808289,
 0.0008380140643566847,
 0.005876685492694378,
 -0.009877001866698265,
 -0.003068646416068077,
 -0.02475765533745289,
 0.01081518642604351,
 -0.013786105439066887,
 -0.024470988661050797,
 -0.014111863449215889,
 0.002454591216519475,
 -0.018998242914676666,
 0.0005786287947557867,
 -0.011336400173604488,
 0.01813824102282524,
 -0.00996169913560152,
 0.013147618621587753,
 -0.011329885572195053,
 -0.00914730224758

In [56]:
from pinecone import ServerlessSpec

os.environ["PINECONE_API_KEY"] = "pcsk_2bwHg3_PWK83jEg965i4tAbAWComBvHu1aPFAMz6BAMnfbvshst4enKbhden7BVz1NnFtZ"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
print(pc.list_indexes())



[{
    "name": "langchainvector",
    "metric": "cosine",
    "host": "langchainvector-7rkl7oj.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}]


In [38]:
vectorstore = PineconeVectorStore.from_documents(
    documents=doc,
    embedding=embeddings,
    index_name="langchainvector",
    pinecone_api_key="pcsk_2bwHg3_PWK83jEg965i4tAbAWComBvHu1aPFAMz6BAMnfbvshst4enKbhden7BVz1NnFtZ"
)

In [57]:
##Retrieve Result
def retrieve_query(query, k=3):
    matching_results = vectorstore.similarity_search(query, k=k)
    return matching_results


In [58]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(api_key=os.environ['OPENAI_API_KEY'], model="gpt-3.5-turbo")
retriever = vectorstore.as_retriever()

In [59]:
# Query the chain
query = """

can you point out where can i can see the information about this question : 


"question": "Which of the following is a correct statement?",
        "options": [
            "A developer makes a mistake which causes a defect that may be seen as a failure during dynamic testing",
            "A developer makes an error which results in a failure that may be seen as a fault when the software is executed",
            "A developer has introduced a failure which results in a defect that may be seen as a mistake during dynamic testing",
            "A developer makes a mistake which causes a bug that may be seen as a defect when the software is executed"
        ],


"""
result = llm.invoke(retriever.invoke(query)[0].page_content + "\n\nQuestion: " + query)
print(result)

content='The correct statement can be found in the section 4.4.1 Error Guessing and Fault Attacks. It states that "A developer makes a mistake which causes a defect that may be seen as a failure during dynamic testing".' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 879, 'total_tokens': 925, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DL0tpma3alAuY02IjHWq3yRJhZBST', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d04b8-d9ae-7661-bb08-8fe53ec811f3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 879, 'output_tokens': 46, 'total_tokens': 925, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'ou